[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/lfm2_chatbot_colab.ipynb)

# 💧 LFM2.5-350M In-Notebook Chatbot

This notebook loads [LiquidAI/LFM2.5-350M-ONNX](https://huggingface.co/LiquidAI/LFM2.5-350M-ONNX) and runs a **chat UI entirely inside Colab** using `google.colab.kernel.invokeFunction` to bridge JS → Python.

No Flask, no tunnels — just direct kernel calls.

## 1 — Install dependencies

In [ ]:
!pip install -q onnxruntime transformers huggingface_hub numpy
!curl -sL https://raw.githubusercontent.com/jonasneves/colab-slm-playground/main/chat_ui.py -o /content/chat_ui.py

## 2 — Download model & tokenizer

In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

MODEL_ID = "LiquidAI/LFM2.5-350M-ONNX"
VARIANT = "model_q4"  # smallest variant (~276MB)

print("Downloading ONNX model...")
model_path = hf_hub_download(MODEL_ID, f"onnx/{VARIANT}.onnx")
data_path  = hf_hub_download(MODEL_ID, f"onnx/{VARIANT}.onnx_data")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Model downloaded to: {model_path}")

## 3 — Load ONNX session & define generation

In [ ]:
import numpy as np
import onnxruntime as ort

print("Loading ONNX session (this may take a moment)...")
session = ort.InferenceSession(model_path)
print(f"Session loaded: {len(session.get_inputs())} inputs, {len(session.get_outputs())} outputs")

TEMPERATURE = 0.7
TOP_K = 50
REPETITION_PENALTY = 1.05
MAX_NEW_TOKENS = 512

ONNX_DTYPE = {
    "tensor(float)": np.float32,
    "tensor(float16)": np.float16,
    "tensor(int64)": np.int64,
}

input_names = {inp.name for inp in session.get_inputs()}
use_position_ids = "position_ids" in input_names


def _init_cache():
    cache = {}
    for inp in session.get_inputs():
        if inp.name in {"input_ids", "attention_mask", "position_ids"}:
            continue
        shape = [d if isinstance(d, int) else 1 for d in inp.shape]
        for i, d in enumerate(inp.shape):
            if isinstance(d, str) and "sequence" in d.lower():
                shape[i] = 0
        cache[inp.name] = np.zeros(shape, dtype=ONNX_DTYPE.get(inp.type, np.float32))
    return cache


def _sample_token(logits, generated_tokens):
    for tid in set(generated_tokens):
        if logits[tid] > 0:
            logits[tid] /= REPETITION_PENALTY
        else:
            logits[tid] *= REPETITION_PENALTY
    logits = logits / TEMPERATURE
    top_k_idx = np.argpartition(logits, -TOP_K)[-TOP_K:]
    top_k_logits = logits[top_k_idx]
    top_k_logits -= np.max(top_k_logits)
    probs = np.exp(top_k_logits) / np.sum(np.exp(top_k_logits))
    chosen = np.random.choice(len(top_k_idx), p=probs)
    return int(top_k_idx[chosen])


def generate(messages: list) -> str:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = np.array(
        [tokenizer.encode(prompt, add_special_tokens=False)], dtype=np.int64
    )
    seq_len = input_ids.shape[1]
    cache = _init_cache()
    generated_tokens = []

    for step in range(MAX_NEW_TOKENS):
        if step == 0:
            ids = input_ids
            pos = np.arange(seq_len, dtype=np.int64).reshape(1, -1)
        else:
            ids = np.array([[generated_tokens[-1]]], dtype=np.int64)
            pos = np.array([[seq_len + len(generated_tokens) - 1]], dtype=np.int64)

        attn_mask = np.ones((1, seq_len + len(generated_tokens)), dtype=np.int64)
        feed = {"input_ids": ids, "attention_mask": attn_mask, **cache}
        if use_position_ids:
            feed["position_ids"] = pos

        outputs = session.run(None, feed)
        logits = outputs[0][0, -1].copy()
        next_token = _sample_token(logits, generated_tokens)
        generated_tokens.append(next_token)

        for i, out in enumerate(session.get_outputs()[1:], 1):
            name = out.name.replace("present_conv", "past_conv").replace(
                "present.", "past_key_values."
            )
            if name in cache:
                cache[name] = outputs[i]

        if next_token == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


print("Session ready. Run the next cell to open the chat.")

# Uncomment to run a quick sanity check before opening the chat:
# reply = generate([{"role": "user", "content": "Hi!"}])
# print(f"Model says: {reply[:200]}")

## 4 — Register callback & launch Chat UI

In [ ]:
from chat_ui import build_chat_html, register_callback
from IPython.display import display, HTML

register_callback(generate)
display(HTML(build_chat_html("LFM2.5-350M", "ONNX &middot; Q4")))
print("Chat ready. Each response may take 10–30 s depending on length.")

---
### Notes

**Why raw ONNX Runtime instead of Optimum?**
LFM2 uses a non-standard KV-cache layout — its past/present key-value tensors don't follow the naming convention that `ORTModelForCausalLM` expects, so Optimum can't drive it. This notebook manages the ONNX session directly: initializing the cache, feeding tokens one at a time, and routing present → past tensors by name between steps.

**Why Q4 by default?**
The model ships four variants: `model_q4` (276 MB), `model_q8` (553 MB), `model_fp16` (692 MB), and `model_fp32` (1.4 GB). Q4 is the smallest and runs on Colab's free CPU tier in 5–15 s per response. For better output quality at the cost of speed, change `VARIANT = "model_fp16"` in cell 2.

**What is LFM?**
LFM (Liquid Foundation Model) is a hybrid architecture from [Liquid AI](https://www.liquid.ai/) combining structured state-space dynamics with attention. Unlike pure transformer models, it maintains a fixed-size recurrent state rather than an ever-growing KV cache — which is why its ONNX export looks different from standard decoder models.

**Multi-turn context**
The full conversation history is sent to the model on every turn. This means response time grows slowly as the conversation gets longer, since the prompt length increases.